In [24]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sharpy.cases.templates.flying_wings as wings
import sharpy.sharpy_main
import json

In [25]:
u_inf = 25 # m/s
alpha_deg = 2
rho = 1.225

In [26]:
number_chordwise_panels = 4
number_spanwise_nodes = 16
wake_length_factor = 10

num_control_surfaces = 2

num_cores = 4
simulation_time =4.0

cases_folder = './cases/' # folder to store input files
output_folder = './output' # folder to save results
route_notebook_dir =  os.path.abspath('')

In [27]:
case_name = "myCaseControlThree"

pazy_model_open_loop = wings.PazyControlSurface(M=number_chordwise_panels,
                                      N=number_spanwise_nodes,
                                      Mstar_fact=wake_length_factor,
                                      u_inf=u_inf,
                                      alpha=alpha_deg,
                                      cs_deflection=[0,0],
                                      n_control_surfaces=2,
                                      rho=rho,
                                      tip_rod = False,
                                      b_ref=2. * 1.2,
                                      main_chord= 0.2,
                                      pct_flap=0.2375, #285mm/1200mm
                                      aspect_ratio=(2. * 1.2) / 0.2,
                                      sweep=0,
                                      cs_type=0, #flap?
                                      n_surfaces=2,
                                      route=cases_folder + '/' + case_name,
                                      case_name=case_name,
                                      physical_time=simulation_time,)

def new_update_mass_stiff(self):
    import sharpy.utils.algebra as algebra

    #seeing our wing has roughly 2% of the area of the goland wing we have to scale the 
    #structural properties appropriately to achieve flutter, (as lift forces are proportional
    #to the wing area)
    width = 100E-3
    height = 2E-3
    E = 69e9  # Young's modulus in Pascals (Pa)
    G = 26e9  # Shear modulus in Pascals (Pa)
    rho = 2700

    # Cross-sectional properties
    A = width * height  # Area in m²
    Iy = (width * height**3) / 12  # Second moment of area about the y-axis (m⁴)
    Iz = (height * width**3) / 12  # Second moment of area about the z-axis (m⁴)
    print(Iy)
    print(Iz)
    J = 2*height**3*width/3  # Torsion constant approximation for rectangular section

    # Rigidity calculations
    EA = E * A
    GA = G * A
    GJ = G * J
    EIy = E * Iy
    EIz = E * Iz

    ea, ga = EA, GA
    gj = GJ
    eiy = EIy
    eiz = EIz
    base_stiffness = np.diag([ea, ga, ga, gj, eiy, eiz])
    self.stiffness = np.zeros((1, 6, 6))
    self.stiffness[0] = base_stiffness
    print(base_stiffness)
    
    m_unit = 1.42 
    j_tors = (Iy+Iz)*rho
    print(Iy+Iz)
    print(j_tors)
    pos_cg_b = np.array([0., self.c_ref * (self.main_cg - self.main_ea), 0.])
    m_chi_cg = algebra.skew(m_unit * pos_cg_b)
    self.mass = np.zeros((1, 6, 6))
    self.mass[0, :, :] = np.diag([m_unit, m_unit, m_unit,
                                    j_tors, .1 * j_tors, .9 * j_tors])

    self.mass[0, :3, 3:] = m_chi_cg
    self.mass[0, 3:, :3] = -m_chi_cg

    self.elem_stiffness = np.zeros((self.num_elem_tot,), dtype=int)
    self.elem_mass = np.zeros((self.num_elem_tot,), dtype=int)

    self.main_ea = 0.33
    self.main_cg = 0.43

pazy_model_open_loop.update_mass_stiff = new_update_mass_stiff.__get__(pazy_model_open_loop, wings.PazyControlSurface) 

In [28]:
def generate_aero_and_fem_input_files(model):
    model.clean_test_files()
    model.update_derived_params()
    model.generate_aero_file()
    model.generate_fem_file()
    
generate_aero_and_fem_input_files(pazy_model_open_loop)

print(pazy_model_open_loop.main_cg)

6.666666666666668e-11
1.666666666666667e-07
[[1.38000000e+07 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 5.20000000e+06 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 5.20000000e+06 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 1.38666667e+01 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 4.60000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 1.15000000e+04]]
1.6673333333333336e-07
0.00045018000000000006
0.43


In [29]:
pazy_model_open_loop.set_default_config_dict()
pazy_model_open_loop.config['SHARPy'] = {
    'flow':
    [
        'BeamLoader',
        'AerogridLoader',
        'StaticCoupled',
        'AerogridPlot',
        'BeamPlot',
        'DynamicCoupled',
    ],
    'case': pazy_model_open_loop.case_name, 'route' : pazy_model_open_loop.route,
    'write_screen': 'on', 'write_log': 'on',
    'save_settings' : 'on',
    'log_folder': output_folder + '/',
    'log_file': pazy_model_open_loop.case_name+'.log'
    }

pazy_model_open_loop.config['BeamLoader'] = {
    'unsteady': 'off',
    'orientation': pazy_model_open_loop.quat
}

pazy_model_open_loop.config['AerogridLoader'] = {
    'unsteady': 'on',
    'aligned_grid': 'on',
    'mstar': wake_length_factor*pazy_model_open_loop.M,
    'wake_shape_generator': 'StraightWake',
    'wake_shape_generator_input': {
        'u_inf':pazy_model_open_loop.u_inf,
        'u_inf_direction': [1., 0., 0.],
        'dt': pazy_model_open_loop.dt,
    },
}

pazy_model_open_loop.config['StaticCoupled'] = {
        'print_info': 'on',
        'max_iter': 200,
        'n_load_steps': 5,
        'tolerance': 1e-5,
        'relaxation_factor': 0.1,
        'aero_solver': 'StaticUvlm',
        'aero_solver_settings': {
            'rho': pazy_model_open_loop.rho,
            'print_info': 'off',
            'horseshoe': 'on',
            'num_cores': num_cores,
            'n_rollup': 0,
            'velocity_field_generator': 'SteadyVelocityField',
            'velocity_field_input': {
                'u_inf': pazy_model_open_loop.u_inf,
                'u_inf_direction': pazy_model_open_loop.u_inf_direction},
            'vortex_radius': 1e-9},
        'structural_solver': 'NonLinearStatic',
        'structural_solver_settings': {'print_info': 'off',
                                   'max_iterations': 200,
                                   'num_load_steps': 5,
                                   'delta_curved': 1e-6,
                                   'min_delta': 1e-8,
                                   'gravity': True,
                                   'gravity': 9.81},
    }
pazy_model_open_loop.config['AerogridPlot'] = {'include_rbm': 'off',
                                               'include_applied_forces': 'on',
                                               'minus_m_star': 0}

pazy_model_open_loop.config['BeamPlot'] = {'include_rbm': 'off',
                                           'include_applied_forces': 'on'}
    
pazy_model_open_loop.config['WriteVariablesTime'] = {'structure_variables': ['pos'],
                                                     'structure_nodes': list(range(0, pazy_model_open_loop.num_node_surf)),
                                                     'cleanup_old_solution': 'on',}

pazy_model_open_loop.config['DynamicCoupled'] = {'print_info': 'on',
                                  'structural_substeps': 10,
                                  'dynamic_relaxation': 'on',
                                  'cleanup_previous_solution': 'on',
                                  'structural_solver': 'NonLinearDynamicPrescribedStep',
                                  'structural_solver_settings':  {'print_info': 'off',
                                                                'max_iterations': 950,
                                                                'delta_curved': 1e-1,
                                                                'min_delta': 1e-6,
                                                                'newmark_damp': 0.5e-4,
                                                                'gravity': True,
                                                                'gravity': 9.81,
                                                                'num_steps': pazy_model_open_loop.n_tstep,
                                                                'dt':pazy_model_open_loop.dt,
                                                                },
                                  'aero_solver': 'StepUvlm',
                                  'aero_solver_settings':  {'print_info': 'on',
                                                            'num_cores': num_cores,
                                                            'convection_scheme': 2,
                                                            'velocity_field_generator': 'SteadyVelocityField',
                                                            'velocity_field_input': {'u_inf': pazy_model_open_loop.u_inf,
                                                                                    'u_inf_direction': [1., 0., 0.]},
                                                            'rho': pazy_model_open_loop.rho,
                                                            'n_time_steps': pazy_model_open_loop.n_tstep,
                                                            'vortex_radius': 1e-9,
                                                            'dt': pazy_model_open_loop.dt,
                                                            'gamma_dot_filtering': 3},
                                    'controller_id': {'controller_tip': 'ControlSurfacePidController'},
                                    'controller_settings': {'controller_tip': {'P': 20.0,
                                                                           'I': 20.0,
                                                                           'D': 20.0,
                                                                           'dt': pazy_model_open_loop.dt,
                                                                           'input_type': 'pitch',
                                                                           #'controller_log_route': './output/' + case_name + '/',
                                                                           'controlled_surfaces': 0,
                                                                           'time_history_input_file': 'pitch.csv'}},
                                  'fsi_substeps': 200,
                                  'fsi_tolerance': 1e-6,
                                  'relaxation_factor': pazy_model_open_loop.relaxation_factor,
                                  'minimum_steps': 1,
                                  'relaxation_steps': 150,
                                  'final_relaxation_factor': 0.0,
                                  'n_time_steps': pazy_model_open_loop.n_tstep,
                                  'dt': pazy_model_open_loop.dt,
                                  'include_unsteady_force_contribution':  True,
                                  'postprocessors': ['WriteVariablesTime', 'BeamPlot', 'AerogridPlot'],
                                  'postprocessors_settings': {'BeamPlot': {'include_rbm': 'on',
                                                                           'include_applied_forces': 'on'},
                                                              'StallCheck': {},
                                                              'AerogridPlot': {
                                                                  'u_inf': pazy_model_open_loop.u_inf,
                                                                  'include_rbm': 'on',
                                                                  'include_applied_forces': 'on',
                                                                  'minus_m_star': 0},
                                                              'WriteVariablesTime': {
                                                                  'structure_variables': ['pos', 'psi'],
                                                                  'structure_nodes': [pazy_model_open_loop.num_node_surf - 1,
                                                                                      pazy_model_open_loop.num_node_surf,
                                                                                      pazy_model_open_loop.num_node_surf + 1],
                                                                    },
                                                                    },
                                    'network_settings': {},
                                    }

pazy_model_open_loop.config['PickleData'] = {}
pazy_model_open_loop.config['LinearAssembler'] = {'linear_system': 'LinearAeroelastic',
                                    'inout_coordinates': 'nodes', 
                                    # 'recover_accelerations': True,           
                                'linear_system_settings': {
                                    'beam_settings': {'modal_projection': True,
                                                        'inout_coords': 'modes',
                                                        'discrete_time': True,
                                                        'newmark_damp': 0.5e-4,
                                                        'discr_method': 'newmark',
                                                        'dt': pazy_model_open_loop.dt,
                                                        'proj_modes': 'undamped',
                                                        'num_modes': 20,
                                                        'print_info': 'on',
                                                        'gravity': True,
                                                        'remove_dofs': []},
                                    'aero_settings': {'dt': pazy_model_open_loop.dt,
                                                        'integr_order': 2,
                                                        'density': pazy_model_open_loop.rho,
                                                        'remove_predictor': True,
                                                        'use_sparse': 'off',
                                                        'gust_assembler':  'LeadingEdge',
                                                        'ScalingDict': {'length':1, 'speed': 1, 'density': 1},
                                                        },
                                    'track_body': False,
                                    'use_euler': False,
                                    }}
pazy_model_open_loop.config['Modal'] = {'print_info': True,
                            'use_undamped_modes': True,
                            'NumLambda': 20,
                            'rigid_body_modes': False, 
                            'write_modes_vtk': False,
                            'print_matrices': False,
                            'continuous_eigenvalues': 'off',
                            'dt': pazy_model_open_loop.dt,
                            'plot_eigenvalues': False,
                            }
pazy_model_open_loop.config['SaveData']['save_linear'] = True

pazy_model_open_loop.config.write()

In [30]:
sharpy.sharpy_main.main(['', pazy_model_open_loop.route + pazy_model_open_loop.case_name + '.sharpy'])

--------------------------------------------------------------------------------
            ######  ##     ##    ###    ########  ########  ##    ##
           ##    ## ##     ##   ## ##   ##     ## ##     ##  ##  ##
           ##       ##     ##  ##   ##  ##     ## ##     ##   ####
            ######  ######### ##     ## ########  ########     ##
                 ## ##     ## ######### ##   ##   ##           ##
           ##    ## ##     ## ##     ## ##    ##  ##           ##
            ######  ##     ## ##     ## ##     ## ##           ##
--------------------------------------------------------------------------------
Aeroelastics Lab, Aeronautics Department.
    Copyright (c), Imperial College London.
    All rights reserved.
    License available at https://github.com/imperialcollegelondon/sharpy
Running SHARPy from /home/user/sharpy/sharpy
SHARPy being run is in /home/user/.local/lib/python3.10/site-packages
SHARPy output folder set
	./output//myCaseControlThree/
Generating an i

fatal: not a git repository (or any of the parent directories): .git


|  3  |  0  |  -6.30984  | -0.2044  |  0.0000  |  6.8295  | -0.0000  |  0.1036  | -0.0000  |
|  0  |  1  |  0.00000   | -0.3904  |  0.0001  | 13.3267  | -0.0001  |  0.1818  | -0.0000  |
|  1  |  1  |  -3.91008  | -0.4008  |  0.0004  | 13.4971  | -0.0005  |  0.1829  | -0.0000  |
|  2  |  1  |  -4.81458  | -0.4021  |  0.0005  | 13.5181  | -0.0006  |  0.1830  | -0.0000  |
|  3  |  1  |  -6.43732  | -0.4021  |  0.0005  | 13.5185  | -0.0006  |  0.1830  | -0.0000  |
|  0  |  2  |  0.00000   | -0.5789  |  0.0011  | 19.4215  | -0.0008  |  0.2327  | -0.0001  |
|  1  |  2  |  -3.80672  | -0.5736  |  0.0029  | 19.3694  | -0.0023  |  0.2338  | -0.0002  |
|  2  |  2  |  -5.67796  | -0.5739  |  0.0030  | 19.3750  | -0.0023  |  0.2339  | -0.0002  |
|  0  |  3  |  0.00000   | -0.7388  |  0.0050  | 24.3279  | -0.0030  |  0.2562  | -0.0003  |
|  1  |  3  |  -3.03706  | -0.7087  |  0.0094  | 24.0472  | -0.0058  |  0.2630  | -0.0005  |
|  2  |  3  |  -4.09838  | -0.7117  |  0.0088  | 24.0811  | -0.0054  |

OSError: File pitch.csv not found in Controller